In [1]:
# ==============================================================
# CUSTOMER CHURN CLASSIFICATION
# ==============================================================
# Algorithms:
# 1. K-Nearest Neighbors (KNN)
# 2. Decision Tree
# 3. Random Forest
# 4. Support Vector Classifier (SVC)
# 5. Logistic Regression
#
# Dataset:
# customer_churn_classification_1000.csv
# ==============================================================


# ==============================================================
# STEP 1: IMPORT LIBRARIES
# ==============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time

# Train-test split
from sklearn.model_selection import train_test_split

# Preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# Classification Algorithms
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

# Evaluation Metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    RocCurveDisplay
)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# ==============================================================
# STEP 2: LOAD DATASET
# ==============================================================
file="https://raw.githubusercontent.com/swapnilsaurav/AIML/refs/heads/main/ML/Classification-CustomerChurnPrediction/customer_churn_classification.csv"
df = pd.read_csv(file)

print("Dataset loaded successfully.")

print("\nDataset Shape:")
print(df.shape)

print("\nFirst 5 Rows:")
print(df.head())

Dataset loaded successfully.

Dataset Shape:
(1000, 15)

First 5 Rows:
  Customer_ID   Age  Gender  Tenure_Months  Monthly_Charges  Total_Charges  \
0    CUST0001  56.0  Female             89            68.23        5610.93   
1    CUST0002  69.0    Male             18           122.79        2187.79   
2    CUST0003  46.0  Female             19            20.00         395.21   
3    CUST0004  32.0  Female             36            99.74        3581.77   
4    CUST0005  60.0    Male             74              NaN        9305.98   

    Contract_Type Internet_Service  Support_Calls  Monthly_Data_GB  \
0        Two Year            Fiber              2             30.8   
1        One Year              DSL              3             53.5   
2  Month-to-Month              NaN              4              1.4   
3        One Year            Fiber              2             20.2   
4        One Year            Fiber              0             16.3   

  Payment_Method Auto_Pay  Satisfaction

In [3]:
# ==============================================================
# STEP 3: UNDERSTAND THE DATA
# ==============================================================

print("\nColumn Names:")
print(df.columns.tolist())

print("\nDataset Information:")
df.info()

print("\nStatistical Summary:")
print(df.describe())

print("\nMissing Values:")
print(df.isnull().sum())

print("\nTarget Distribution:")
print(df["Churn"].value_counts())

print("\nTarget Distribution (%):")
print(df["Churn"].value_counts(normalize=True) * 100)


Column Names:
['Customer_ID', 'Age', 'Gender', 'Tenure_Months', 'Monthly_Charges', 'Total_Charges', 'Contract_Type', 'Internet_Service', 'Support_Calls', 'Monthly_Data_GB', 'Payment_Method', 'Auto_Pay', 'Satisfaction_Score', 'Late_Payments', 'Churn']

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Customer_ID         1000 non-null   object 
 1   Age                 980 non-null    float64
 2   Gender              1000 non-null   object 
 3   Tenure_Months       1000 non-null   int64  
 4   Monthly_Charges     980 non-null    float64
 5   Total_Charges       975 non-null    float64
 6   Contract_Type       1000 non-null   object 
 7   Internet_Service    902 non-null    object 
 8   Support_Calls       1000 non-null   int64  
 9   Monthly_Data_GB     1000 non-null   float64
 10  Payment_Method      985 non

In [4]:
# ==============================================================
# STEP 4: CHECK DUPLICATES
# ==============================================================

print("Number of duplicate rows:")
print(df.duplicated().sum())

#If duplicates exist:
'''
df = df.drop_duplicates()

print("Shape after removing duplicates:")
print(df.shape)
'''
# ==============================================================
# STEP 5: DROP IDENTIFIER COLUMN
# ==============================================================

df = df.drop(columns=["Customer_ID"])

print(df.head())


Number of duplicate rows:
0
    Age  Gender  Tenure_Months  Monthly_Charges  Total_Charges  \
0  56.0  Female             89            68.23        5610.93   
1  69.0    Male             18           122.79        2187.79   
2  46.0  Female             19            20.00         395.21   
3  32.0  Female             36            99.74        3581.77   
4  60.0    Male             74              NaN        9305.98   

    Contract_Type Internet_Service  Support_Calls  Monthly_Data_GB  \
0        Two Year            Fiber              2             30.8   
1        One Year              DSL              3             53.5   
2  Month-to-Month              NaN              4              1.4   
3        One Year            Fiber              2             20.2   
4        One Year            Fiber              0             16.3   

  Payment_Method Auto_Pay  Satisfaction_Score  Late_Payments Churn  
0  Bank Transfer      Yes                 3.0              0    No  
1            UPI

In [7]:
# ==============================================================
# STEP 6: CREATE FEATURES X AND TARGET y
# ==============================================================

X = df.drop(columns=["Churn"])

y = df["Churn"]

print("X Shape:", X.shape)
print("y Shape:", y.shape)
# ==============================================================
# STEP 7: ENCODE TARGET
# ==============================================================

y = y.map({
    "No": 0,
    "Yes": 1
})

print(y.head())

print("\nTarget Distribution:")
print(y.value_counts())


X Shape: (1000, 13)
y Shape: (1000,)
0    0
1    0
2    0
3    0
4    1
Name: Churn, dtype: int64

Target Distribution:
Churn
0    792
1    208
Name: count, dtype: int64


In [8]:
# ==============================================================
# STEP 8: IDENTIFY COLUMN TYPES
# ==============================================================

numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()


print("Numeric Features:")
print(numeric_features)

print("\nCategorical Features:")
print(categorical_features)

# ==============================================================
# STEP 9: TRAIN-TEST SPLIT
# ==============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


print("Training X:", X_train.shape)
print("Testing X :", X_test.shape)

print("Training y:", y_train.shape)
print("Testing y :", y_test.shape)


print("\nTraining Target Distribution:")
print(y_train.value_counts())

print("\nTesting Target Distribution:")
print(y_test.value_counts())

Numeric Features:
['Age', 'Tenure_Months', 'Monthly_Charges', 'Total_Charges', 'Support_Calls', 'Monthly_Data_GB', 'Satisfaction_Score', 'Late_Payments']

Categorical Features:
['Gender', 'Contract_Type', 'Internet_Service', 'Payment_Method', 'Auto_Pay']
Training X: (800, 13)
Testing X : (200, 13)
Training y: (800,)
Testing y : (200,)

Training Target Distribution:
Churn
0    634
1    166
Name: count, dtype: int64

Testing Target Distribution:
Churn
0    158
1     42
Name: count, dtype: int64


In [9]:
# ==============================================================
# STEP 10: NUMERIC PREPROCESSOR
# ==============================================================

numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),

        (
            "scaler",
            StandardScaler()
        )
    ]
)

# ==============================================================
# STEP 11: CATEGORICAL PREPROCESSOR
# ==============================================================

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),

        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

# ==============================================================
# STEP 12: COLUMN TRANSFORMER
# ==============================================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_transformer,
            numeric_features
        ),

        (
            "categorical",
            categorical_transformer,
            categorical_features
        )
    ]
)

In [11]:
# ==============================================================
# MODEL 1: K-NEAREST NEIGHBORS
# ==============================================================

knn_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),

        (
            "classifier",
            KNeighborsClassifier(
                n_neighbors=5
            )
        )
    ]
)

# --------------------------------------------------------------
# TRAIN KNN
# --------------------------------------------------------------

start_time = time.perf_counter()

knn_pipeline.fit(
    X_train,
    y_train
)

knn_training_time = time.perf_counter() - start_time


print(
    "KNN Training Time:",
    round(knn_training_time, 6),
    "seconds"
)

# --------------------------------------------------------------
# KNN PREDICTION
# --------------------------------------------------------------

start_time = time.perf_counter()

knn_pred = knn_pipeline.predict(X_test)

knn_prediction_time = time.perf_counter() - start_time


print(
    "KNN Prediction Time:",
    round(knn_prediction_time, 6),
    "seconds"
)

# --------------------------------------------------------------
# KNN EVALUATION
# --------------------------------------------------------------

knn_accuracy = accuracy_score(
    y_test,
    knn_pred
)

knn_precision = precision_score(
    y_test,
    knn_pred
)

knn_recall = recall_score(
    y_test,
    knn_pred
)

knn_f1 = f1_score(
    y_test,
    knn_pred
)


print("\nKNN PERFORMANCE")

print("Accuracy :", knn_accuracy)
print("Precision:", knn_precision)
print("Recall   :", knn_recall)
print("F1 Score :", knn_f1)


print("\nConfusion Matrix:")

print(
    confusion_matrix(
        y_test,
        knn_pred
    )
)


print("\nClassification Report:")

print(
    classification_report(
        y_test,
        knn_pred
    )
)

KNN Training Time: 0.022067 seconds
KNN Prediction Time: 2.409757 seconds

KNN PERFORMANCE
Accuracy : 0.785
Precision: 0.48
Recall   : 0.2857142857142857
F1 Score : 0.3582089552238806

Confusion Matrix:
[[145  13]
 [ 30  12]]

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.92      0.87       158
           1       0.48      0.29      0.36        42

    accuracy                           0.79       200
   macro avg       0.65      0.60      0.61       200
weighted avg       0.76      0.79      0.76       200



In [12]:
# ==============================================================
# MODEL 2: DECISION TREE
# ==============================================================

dt_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),

        (
            "classifier",
            DecisionTreeClassifier(
                max_depth=5,
                random_state=42
            )
        )
    ]
)

start_time = time.perf_counter()

dt_pipeline.fit(
    X_train,
    y_train
)

dt_training_time = time.perf_counter() - start_time


print(
    "Decision Tree Training Time:",
    round(dt_training_time, 6),
    "seconds"
)

start_time = time.perf_counter()

dt_pred = dt_pipeline.predict(X_test)

dt_prediction_time = time.perf_counter() - start_time


print(
    "Decision Tree Prediction Time:",
    round(dt_prediction_time, 6),
    "seconds"
)

dt_accuracy = accuracy_score(
    y_test,
    dt_pred
)

dt_precision = precision_score(
    y_test,
    dt_pred
)

dt_recall = recall_score(
    y_test,
    dt_pred
)

dt_f1 = f1_score(
    y_test,
    dt_pred
)


print("\nDECISION TREE PERFORMANCE")

print("Accuracy :", dt_accuracy)
print("Precision:", dt_precision)
print("Recall   :", dt_recall)
print("F1 Score :", dt_f1)


print("\nConfusion Matrix:")

print(
    confusion_matrix(
        y_test,
        dt_pred
    )
)


print("\nClassification Report:")

print(
    classification_report(
        y_test,
        dt_pred
    )
)

Decision Tree Training Time: 0.028844 seconds
Decision Tree Prediction Time: 0.015392 seconds

DECISION TREE PERFORMANCE
Accuracy : 0.74
Precision: 0.32142857142857145
Recall   : 0.21428571428571427
F1 Score : 0.2571428571428571

Confusion Matrix:
[[139  19]
 [ 33   9]]

Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.88      0.84       158
           1       0.32      0.21      0.26        42

    accuracy                           0.74       200
   macro avg       0.56      0.55      0.55       200
weighted avg       0.71      0.74      0.72       200



In [13]:
# ==============================================================
# MODEL 3: RANDOM FOREST
# ==============================================================

rf_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),

        (
            "classifier",
            RandomForestClassifier(
                n_estimators=200,
                max_depth=8,
                random_state=42,
                class_weight="balanced"
            )
        )
    ]
)

start_time = time.perf_counter()

rf_pipeline.fit(
    X_train,
    y_train
)

rf_training_time = time.perf_counter() - start_time


print(
    "Random Forest Training Time:",
    round(rf_training_time, 6),
    "seconds"
)

start_time = time.perf_counter()

rf_pred = rf_pipeline.predict(X_test)

rf_prediction_time = time.perf_counter() - start_time


print(
    "Random Forest Prediction Time:",
    round(rf_prediction_time, 6),
    "seconds"
)


rf_accuracy = accuracy_score(
    y_test,
    rf_pred
)

rf_precision = precision_score(
    y_test,
    rf_pred
)

rf_recall = recall_score(
    y_test,
    rf_pred
)

rf_f1 = f1_score(
    y_test,
    rf_pred
)


print("\nRANDOM FOREST PERFORMANCE")

print("Accuracy :", rf_accuracy)
print("Precision:", rf_precision)
print("Recall   :", rf_recall)
print("F1 Score :", rf_f1)


print("\nConfusion Matrix:")

print(
    confusion_matrix(
        y_test,
        rf_pred
    )
)


print("\nClassification Report:")

print(
    classification_report(
        y_test,
        rf_pred
    )
)



Random Forest Training Time: 0.715922 seconds
Random Forest Prediction Time: 0.034118 seconds

RANDOM FOREST PERFORMANCE
Accuracy : 0.76
Precision: 0.4117647058823529
Recall   : 0.3333333333333333
F1 Score : 0.3684210526315789

Confusion Matrix:
[[138  20]
 [ 28  14]]

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.87      0.85       158
           1       0.41      0.33      0.37        42

    accuracy                           0.76       200
   macro avg       0.62      0.60      0.61       200
weighted avg       0.74      0.76      0.75       200



In [14]:
# ==============================================================
# MODEL 4: SUPPORT VECTOR CLASSIFIER
# ==============================================================

svc_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),

        (
            "classifier",
            SVC(
                kernel="rbf",
                C=1.0,
                probability=True,
                class_weight="balanced",
                random_state=42
            )
        )
    ]
)

start_time = time.perf_counter()

svc_pipeline.fit(
    X_train,
    y_train
)

svc_training_time = time.perf_counter() - start_time


print(
    "SVC Training Time:",
    round(svc_training_time, 6),
    "seconds"
)

start_time = time.perf_counter()

svc_pred = svc_pipeline.predict(X_test)

svc_prediction_time = time.perf_counter() - start_time


print(
    "SVC Prediction Time:",
    round(svc_prediction_time, 6),
    "seconds"
)

svc_accuracy = accuracy_score(
    y_test,
    svc_pred
)

svc_precision = precision_score(
    y_test,
    svc_pred
)

svc_recall = recall_score(
    y_test,
    svc_pred
)

svc_f1 = f1_score(
    y_test,
    svc_pred
)


print("\nSVC PERFORMANCE")

print("Accuracy :", svc_accuracy)
print("Precision:", svc_precision)
print("Recall   :", svc_recall)
print("F1 Score :", svc_f1)


print("\nConfusion Matrix:")

print(
    confusion_matrix(
        y_test,
        svc_pred
    )
)


print("\nClassification Report:")

print(
    classification_report(
        y_test,
        svc_pred
    )
)

SVC Training Time: 0.423135 seconds
SVC Prediction Time: 0.021341 seconds

SVC PERFORMANCE
Accuracy : 0.73
Precision: 0.4189189189189189
Recall   : 0.7380952380952381
F1 Score : 0.5344827586206896

Confusion Matrix:
[[115  43]
 [ 11  31]]

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.73      0.81       158
           1       0.42      0.74      0.53        42

    accuracy                           0.73       200
   macro avg       0.67      0.73      0.67       200
weighted avg       0.81      0.73      0.75       200



In [15]:
# ==============================================================
# MODEL 5: LOGISTIC REGRESSION
# ==============================================================

lr_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),

        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=42
            )
        )
    ]
)

start_time = time.perf_counter()

lr_pipeline.fit(
    X_train,
    y_train
)

lr_training_time = time.perf_counter() - start_time


print(
    "Logistic Regression Training Time:",
    round(lr_training_time, 6),
    "seconds"
)

start_time = time.perf_counter()

lr_pred = lr_pipeline.predict(X_test)

lr_prediction_time = time.perf_counter() - start_time


print(
    "Logistic Regression Prediction Time:",
    round(lr_prediction_time, 6),
    "seconds"
)

lr_accuracy = accuracy_score(
    y_test,
    lr_pred
)

lr_precision = precision_score(
    y_test,
    lr_pred
)

lr_recall = recall_score(
    y_test,
    lr_pred
)

lr_f1 = f1_score(
    y_test,
    lr_pred
)


print("\nLOGISTIC REGRESSION PERFORMANCE")

print("Accuracy :", lr_accuracy)
print("Precision:", lr_precision)
print("Recall   :", lr_recall)
print("F1 Score :", lr_f1)


print("\nConfusion Matrix:")

print(
    confusion_matrix(
        y_test,
        lr_pred
    )
)


print("\nClassification Report:")

print(
    classification_report(
        y_test,
        lr_pred
    )
)

Logistic Regression Training Time: 0.03649 seconds
Logistic Regression Prediction Time: 0.010056 seconds

LOGISTIC REGRESSION PERFORMANCE
Accuracy : 0.685
Precision: 0.37037037037037035
Recall   : 0.7142857142857143
F1 Score : 0.4878048780487805

Confusion Matrix:
[[107  51]
 [ 12  30]]

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.68      0.77       158
           1       0.37      0.71      0.49        42

    accuracy                           0.69       200
   macro avg       0.63      0.70      0.63       200
weighted avg       0.79      0.69      0.71       200



In [19]:
# ==============================================================
# STEP 14: ROC-AUC SCORES
# ==============================================================

knn_prob = knn_pipeline.predict_proba(X_test)[:, 1]

dt_prob = dt_pipeline.predict_proba(X_test)[:, 1]

rf_prob = rf_pipeline.predict_proba(X_test)[:, 1]

svc_prob = svc_pipeline.predict_proba(X_test)[:, 1]

lr_prob = lr_pipeline.predict_proba(X_test)[:, 1]


knn_auc = roc_auc_score(y_test, knn_prob)

dt_auc = roc_auc_score(y_test, dt_prob)

rf_auc = roc_auc_score(y_test, rf_prob)

svc_auc = roc_auc_score(y_test, svc_prob)

lr_auc = roc_auc_score(y_test, lr_prob)


print("ROC-AUC Scores")

print("KNN                 :", knn_auc)
print("Decision Tree       :", dt_auc)
print("Random Forest       :", rf_auc)
print("SVC                 :", svc_auc)
print("Logistic Regression :", lr_auc)

ROC-AUC Scores
KNN                 : 0.6790988547317661
Decision Tree       : 0.661618444846293
Random Forest       : 0.752411091018686
SVC                 : 0.7721518987341772
Logistic Regression : 0.7542194092827005


In [18]:
# ==============================================================
# STEP 16: BEST MODEL
# ==============================================================

best_model = results.iloc[0]

print("BEST MODEL")
print("-----------------------")

print(
    "Model:",
    best_model["Model"]
)

print(
    "Accuracy:",
    round(best_model["Accuracy"], 4)
)

print(
    "Precision:",
    round(best_model["Precision"], 4)
)

print(
    "Recall:",
    round(best_model["Recall"], 4)
)

print(
    "F1 Score:",
    round(best_model["F1_Score"], 4)
)

print(
    "ROC-AUC:",
    round(best_model["ROC_AUC"], 4)
)

NameError: name 'results' is not defined